#### 09. Final models and the test year

The settings chosen on 2023 in notebook 07 are fixed here. Both models are refitted on 2017–2023, the full training table plus the validation year with every row at weight 1, and so are the threshold mapper and the baselines. Everything is then scored once on 2024, the test year, which no earlier notebook has used.

**Setup**

In [ ]:
%load_ext autoreload
%autoreload 2
import time
import sys
sys.path.insert(0, "../code")
import pandas as pd
from pyspark.sql import functions as F

import config as C
import evaluate as E
import mapper as M
import models as MD
import baselines as B

spark = C.get_spark("task2-final")

**Final settings**

Taken from notebook 07 (LR tuning and random forest screening on 2023) and from notebook 04 for the climatology shrinkage.

In [ ]:
TIERS = ("t0", "t1", "t2", "t3")
LR_SPEC = {"reg_param": 1e-6, "nonlinear": True}
RF_SPEC = {"max_depth": 24, "min_instances": 50, "feature_subset": "onethird", "max_bins": 256, "impurity": "entropy"}
NUM_TREES = C.RF_TREES
CLIM_M = 3000

**Data**

In [ ]:
train_full = spark.read.parquet(str(C.ASSEMBLED / "train_full"))
val = spark.read.parquet(str(C.ASSEMBLED / "val"))
fit = train_full.unionByName(val).cache()
test = spark.read.parquet(str(C.ASSEMBLED / "test")).cache()
FEATURES = MD.feature_columns(fit, TIERS)
print(f"fit 2017-{C.VAL_YEAR}: {fit.count():,} rows   test {C.TEST_YEAR}: {test.count():,} rows   {len(FEATURES)} features")

**Threshold mapper**

In [ ]:
mp = M.fit(fit)
M.save(mp, C.MODELS / "mapper.json")
{k: (len(v["y"]), max(v["y"])) for k, v in mp["classes"].items()}

**Final models**

In [ ]:
final = {}
for kind, build in [("LR", lambda: MD.build_lr(FEATURES, **LR_SPEC)),
                    ("RF", lambda: MD.build_rf(FEATURES, num_trees=NUM_TREES, **RF_SPEC))]:
    start = time.time()
    final[kind] = build().fit(fit)
    final[kind].write().overwrite().save(str(C.MODELS / kind.lower()))
    print(kind, f"fitted and saved in {(time.time() - start) / 60:.0f} min")
print("LR iterations:", final["LR"].stages[-1].summary.totalIterations, "  RF nodes:", final["RF"].stages[-1].totalNumNodes)

**Baselines on the same years**

The constant forecast takes the class shares of 2017–2023, and climatology and persistence are refitted on the same table, which holds every labelled row of those years at weight 1.

In [ ]:
shares = fit.groupBy("label").agg(F.sum("w").alias("sw")).toPandas().set_index("label").sw.sort_index()
shares = shares / shares.sum()
from pyspark.ml.functions import array_to_vector
forecasts = {
    "constant": test.withColumn("probability", array_to_vector(F.array(*[F.lit(float(p)) for p in shares]))),
    "climatology": B.predict_climatology(B.fit_climatology(fit, CLIM_M), test),
    "persistence": B.predict_persistence(B.fit_persistence(fit), test),
    **{f"{kind} {'+'.join(t.upper() for t in TIERS)}": model.transform(test) for kind, model in final.items()},
}

**Results on 2024**

In [ ]:
rows = []
for name, fc in forecasts.items():
    fc = fc.select("label", "y_mm", "w", "regime", "probability").cache()
    rows.append({"model": name, "subset": "all", **E.score(fc)})
    for regime in ["dry_now", "wet_now"]:
        rows.append({"model": name, "subset": regime, **E.score(fc.filter(F.col("regime") == regime))})
    fc.unpersist()
results = pd.DataFrame(rows).set_index(["subset", "model"]).sort_index()
results.to_csv(C.RESULTS / "models_test.csv")
results.loc["all"].sort_values("RPS").round(4)

In [ ]:
cols = ["RPS", "log loss", "Brier 0", "AUC-PR 0", "AUC-PR 5"]
pd.concat({"dry now": results.loc["dry_now"][cols], "wet now": results.loc["wet_now"][cols]}, axis=1).sort_values(("dry now", "RPS")).round(4)

**Thresholds between the class edges**

The threshold mapper turns the class probabilities into P(Y > θ) for any θ. The Brier score at 3, 15 and 30 mm, which are not class edges, checks it on the test year.

In [ ]:
brier = {}
for name, fc in forecasts.items():
    df = fc.select("y_mm", "w", "probability")
    for theta in C.EXTRA_THRESHOLDS:
        df = M.add_threshold(df, mp, theta)
    df = df.cache()
    brier[name] = {f"Brier {t:g}": E.brier_at(df, t) for t in C.EXTRA_THRESHOLDS}
    df.unpersist()
pd.DataFrame(brier).T.round(5)

**Calibration on 2024**

In [ ]:
tables = {}
for kind, model in final.items():
    ex = E.add_exceedance(model.transform(test))
    for th in [0.0, 5.0]:
        tables[f"{kind}, θ = {th:g} mm"] = E.reliability(ex, th).set_index("bin")[["n", "mean_forecast", "observed"]]
pd.concat(tables, axis=1).round(3)

**Feature importance of the final forest**

In [ ]:
importance = pd.Series(final["RF"].stages[-1].featureImportances.toArray(), index=FEATURES, name="importance")
importance.sort_values(ascending=False).head(15).round(4)